# Модуль 1 · МОИ ОТВЕТЫ НА ПРАКТИКУ 
**Первый раз вообще без ИИ для подсказок сделано**


In [1]:
import numpy as np
import pandas as pd

## Данные: orders.csv после очистки урока 6 и customers.csv
Очистку урока 6 собрали в одну ячейку. Проверка: 5 280 строк.

In [6]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv"), "| customers.csv:", os.path.exists("/Users/gulsatkalbayeva/Documents/dsa-homework/customers.csv"))

Положите orders.csv и customers.csv в одну папку с ноутбуком
orders.csv: True | customers.csv: True


In [7]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [27]:
df["cancel"] = df["status"] == "отменён"     # True/False: пригодится для доли отмен
paid = df[df["status"] == "оплачен"]
len(paid)

4739

In [28]:
df

,order_id,order_date,customer_id,city,category,amount,delivery_days,status,month,cancel
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен,9,False
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен,9,False
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен,9,False
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен,9,False
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен,9,False
...,...,...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен,10,False
5459,104648,2026-10-31 21:00:00,C00629,Караганда,Книги,3640.0,2.0,оплачен,10,False
5460,104615,2026-10-31 22:00:00,C00557,Астана,Электроника,22150.0,NaN,оплачен,10,False
5461,103460,2026-10-31 22:00:00,C02162,Алматы,Электроника,24670.0,2.0,отменён,10,True


## Практика 1, 12 минут
1. **[База]** Сколько заказов и какая выручка оплаченных в каждой категории? Отсортируйте по выручке. Сходится с ДЗ 4 (лидер Электроника, 34,4 млн ₸)?
2. **[База]** Таблица по городам: число заказов и доля отмен (`agg` с именами). Какой город выделяется и на скольких заказах это видно?
3. **[Стандарт]** Средний и медианный чек оплаченных заказов по городам (`agg(["mean", "median"])`). Может, Шымкент падает, потому что там покупают дешевле?
4. **[Стандарт]** Доля отмен по сроку доставки: `groupby("delivery_days")`, рядом число заказов. Как отмены зависят от срока?
5. **[Челлендж]** Средний срок доставки по городам и месяцам (`groupby` по двум столбцам + `unstack`). Какая клетка выделяется? Почему `count` и `size` в этой таблице дают разные числа?

In [ ]:
#1**[База]** Сколько заказов и какая выручка оплаченных в каждой категории? 
# Отсортируйте по выручке. Сходится с ДЗ 4 (лидер Электроника, 34,4 млн ₸)?

paid.groupby('category')['amount'].agg(['count', 'sum']).sort_values('sum', ascending=False)

,count,sum
category,,
Электроника,1036,34370430.0
Одежда,1330,17672830.0
Дом,952,10123520.0
Красота,857,5289940.0
Книги,564,2206210.0


In [60]:
# 2 **[База]** Таблица по городам: число заказов и доля отмен (`agg` с именами). 
# Какой город выделяется и на скольких заказах это видно? - Шымкент 776 заказов и отмены 20.6%
df.groupby('city')['cancel'].agg(['count', 'mean']).sort_values('count', ascending=False)

,count,mean
city,,
Алматы,2116,0.077505
Астана,1339,0.088125
Шымкент,776,0.206186
Караганда,519,0.090559
Актобе,387,0.105943
Атырау,143,0.076923


In [ ]:
# 3 **[Стандарт]** Средний и медианный чек оплаченных заказов по городам (`agg(["mean", "median"])`). 
# Может, Шымкент падает, потому что там покупают дешевле?
paid.groupby('city')['amount'].agg(['mean', 'median'])

,mean,median
city,,
Актобе,14790.115607,10210.0
Алматы,14930.373975,10445.0
Астана,14647.624898,10120.0
Атырау,14347.348485,9920.0
Караганда,14873.305085,10455.0
Шымкент,13965.357143,9995.0


In [ ]:
# 4 **[Стандарт]** Доля отмен по сроку доставки: `groupby("delivery_days")`, рядом число заказов. 
# Как отмены зависят от срока? чем дольше доставка тем выше доля отказа 

df.groupby('delivery_days')['cancel'].agg(['count', 'mean'])

,count,mean
delivery_days,,
1.0,1557,0.042389
2.0,1570,0.079618
3.0,1590,0.133962
5.0,86,0.220930
6.0,120,0.216667
7.0,95,0.347368
8.0,104,0.432692


In [55]:
# 5 **[Челлендж]** Средний срок доставки по городам и месяцам (`groupby` по двум столбцам + `unstack`). 
# Какая клетка выделяется? Почему `count` и `size` в этой таблице дают разные числа
# `count` не берет NaN а `size` их включает

df.groupby(['city', 'month'])['delivery_days'].agg(['count', 'size', 'mean']).unstack().round(2)

count        size        mean      
month        9     10    9     10    9     10
city                                         
Актобе      188   195   190   197  2.04  2.10
Алматы     1007  1046  1043  1073  1.98  2.01
Астана      644   655   665   674  2.05  1.99
Атырау       66    72    69    74  2.06  2.01
Караганда   261   234   274   245  1.90  1.99
Шымкент     349   405   359   417  2.03  6.54

## Практика 2, 20 минут
1. **[База]** Таблица «город × месяц»: выручка оплаченных, изменение в % и вклад в падение. Совпадает с уроком 4?
2. **[База]** Таблица «категория × месяц»: выручка оплаченных и изменение в %. Какая категория падает сильнее всех? Сравните с ДЗ 4.
3. **[Стандарт]** Шымкент против остальных городов: число заказов, доля отмен и средний чек по месяцам. Подсказка: `df["is_sh"] = df["city"] == "Шымкент"` и `groupby(["is_sh", "month"])`. Что изменилось в октябре: заказы, отмены или чек?
4. **[Стандарт]** Средний срок доставки «город × месяц» через `pivot_table`. Какая клетка выделяется? Напишите вывод для менеджера одной фразой.
5. **[Челлендж]** Шымкент по дням: средний срок доставки (`groupby` по `.dt.date`). Найдите первый день, когда средний срок не меньше 5. Проверьте, что в остальных городах в этот день срок обычный.

In [74]:
# 1 Таблица «город × месяц»: выручка оплаченных, изменение в % и вклад в падение. 
# Совпадает с уроком 4?
p = paid.pivot_table(values='amount', index='city', columns='month', aggfunc='sum')
p['change_%'] = (p[10] / p[9] - 1) * 100
p['diff'] = p[10] - p[9]
p["share_of_drop_%"] = p['diff']/ p["diff"].sum() * 100
p.sort_values("diff").round(1)

month,9,10,change_%,diff,share_of_drop_%
city,,,,,
Шымкент,4558650.0,4044010.0,-11.3,-514640.0,62.5
Караганда,3648520.0,3371680.0,-7.6,-276840.0,33.6
Астана,9034220.0,8850530.0,-2.0,-183690.0,22.3
Актобе,2576960.0,2540420.0,-1.4,-36540.0,4.4
Атырау,923990.0,969860.0,5.0,45870.0,-5.6
Алматы,14501070.0,14643020.0,1.0,141950.0,-17.2


In [ ]:
# **[База]** Таблица «категория × месяц»: выручка оплаченных и изменение в %. 
# Какая категория падает сильнее всех? Сравните с ДЗ 4. Одежда	-7.2
c = paid.pivot_table(values='amount', index='category', columns='month', aggfunc='sum')
c['change_%'] = (c[10]/c[9]-1)*100
c.sort_values('change_%').round(1)

month,9,10,change_%
category,,,
Одежда,9167100.0,8505730.0,-7.2
Электроника,17402330.0,16968100.0,-2.5
Дом,4997730.0,5125790.0,2.6
Красота,2610950.0,2678990.0,2.6
Книги,1065300.0,1140910.0,7.1


In [109]:
# 3 Шымкент против остальных городов: число заказов, доля отмен и средний чек по месяцам. Подсказка: `df["is_sh"] = df["city"] == "Шымкент"` и `groupby(["is_sh", "month"])`. 
# Что изменилось в октябре: заказы, отмены или чек? отмены cтало больше 

df["is_sh"] = df["city"] == "Шымкент"
s = df.groupby(["is_sh", "month"]).agg(count=('amount', 'count'), 
                                   cancel_rate = ('cancel', 'mean'))

paid["is_sh"] = paid["city"] == "Шымкент"
s['average'] = paid.groupby(["is_sh", "month"]).agg(average = ('amount', 'mean'))
s

count  cancel_rate       average
is_sh month                                  
False 9       2241     0.087461  15004.772616
      10      2263     0.081750  14617.666025
True  9        359     0.089136  13940.825688
      10       417     0.306954  13993.114187

In [118]:
# 4 Средний срок доставки «город × месяц» через `pivot_table`. 
# Какая клетка выделяется? Напишите вывод для менеджера одной фразой.
print("В Шымкенте средний срок доставки вырос с 2.03 в сентябре до 6.54 в ноябре")
df.pivot_table(values='delivery_days', index='city', columns='month', aggfunc='mean').round(2)


В Шымкенте средний срок доставки вырос с 2.03 в сентябре до 6.54 в ноябре


month,9,10
city,,
Актобе,2.04,2.10
Алматы,1.98,2.01
Астана,2.05,1.99
Атырау,2.06,2.01
Караганда,1.90,1.99
Шымкент,2.03,6.54


In [143]:
# 5 Шымкент по дням: средний срок доставки (`groupby` по `.dt.date`). 
# Найдите первый день, когда средний срок не меньше 5. 
# Проверьте, что в остальных городах в этот день срок обычный.
df["day"] = df["order_date"].dt.date
shym = df[df['city']=='Шымкент']
sh_days = shym.groupby("day")["delivery_days"].mean()
first_day = sh_days[sh_days>=5].index[0]
print('Первый день, когда срок больше пяти дней', first_day, sh_days[first_day])

other = df[df['city']!='Шымкент']
other_days = other.groupby("day")["delivery_days"].mean()
print('Тем временем в других городах:',other_days[first_day])

Первый день, когда срок больше пяти дней 2026-10-01 6.214285714285714
Тем временем в других городах: 1.8888888888888888


## 3. merge: заказы + клиенты (06-Merging)
Маркетолог говорит: «В октябре упала органика, проблема в SEO». Канала привлечения нет в orders.csv: он в customers.csv. `merge` склеивает две таблицы по общему столбцу, как ВПР в Excel.

MERGE, JOIN, CONCAT

In [154]:
cu = pd.read_csv("/Users/gulsatkalbayeva/Documents/dsa-homework/customers.csv")
cu.shape, cu["customer_id"].is_unique

((2200, 4), True)

In [155]:
cu.head()

,customer_id,channel,age,signup_date
0,C00001,рефералы,31,2026-08-06
1,C00002,реклама,30,2025-07-16
2,C00003,органика,45,2026-05-22
3,C00004,органика,43,2025-10-11
4,C00005,реклама,35,2025-09-02


In [157]:
m = df.merge(cu, on="customer_id", how="left")
len(df), len(m), m["channel"].isna().sum()

(5280, 5280, np.int64(0))

In [158]:
m.head()

,order_id,order_date,customer_id,city,category,amount,delivery_days,status,month,cancel,is_sh,day,channel,age,signup_date
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен,9,False,False,2026-09-01,рефералы,35,2025-02-11
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен,9,False,False,2026-09-01,органика,45,2025-07-18
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен,9,False,False,2026-09-01,органика,22,2025-05-05
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен,9,False,False,2026-09-01,органика,30,2026-05-10
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен,9,False,False,2026-09-01,рефералы,45,2025-12-05


Органика −6,0%. Значит, проблема в SEO? Проверим в Практике 3.

## Практика 3, 15 минут
1. **[База]** Сколько строк до и после `merge`? Сколько заказов без канала? Сколько клиентов из customers.csv не сделали ни одного заказа в сентябре–октябре (`isin`)?
2. **[База]** Выручка оплаченных по каналам и месяцам, изменение в %.
3. **[Стандарт]** Проверьте гипотезу маркетолога: по каналам и месяцам число оплаченных заказов, средний чек и доля отмен. Какая доля заказов каждого канала из Шымкента? Объясняет ли канал падение?
4. **[Челлендж]** Сделайте `merge` по умолчанию (`inner`) с исходным `pd.read_csv("orders.csv")`. Сколько строк пропало, какие и почему код не предупредил? Как это поймать проверкой?

In [171]:
# 1 Сколько строк до и после `merge`? 
# Сколько заказов без канала? 
# Сколько клиентов из customers.csv не сделали ни одного заказа в сентябре–октябре (`isin`)?

print("ДО: ", df.shape, cu.shape, "После:", m.shape)
print(m["channel"].isna().sum(), "заказов без канала")

no_orders = ~cu["customer_id"].isin(df["customer_id"])
print(no_orders.sum(), "не сделали заказ")

ДО:  (5280, 12) (2200, 4) После: (5280, 15)
0 заказов без канала
198 не сделали заказ


In [186]:
# 2 Выручка оплаченных по каналам и месяцам, изменение в %. 
m_paid = paid.merge(cu, on="customer_id", how="left")

channel = m_paid.pivot_table(values='amount', index='channel', columns='month', aggfunc='sum')
channel['change_%']=(channel[10]/channel[9]-1)*100
channel.sort_values('change_%',ascending=False).round(1)

month,9,10,change_%
channel,,,
рефералы,4996010.0,5467900.0,9.4
реклама,18034980.0,17467600.0,-3.1
органика,12212420.0,11484020.0,-6.0


In [ ]:
# 3 Проверьте гипотезу маркетолога: 
# по каналам и месяцам число оплаченных заказов, средний чек и доля отмен. 
ch_m= m_paid.groupby(['channel', 'month']).agg(count=('amount', 'count'), average_check=('amount', 'mean'))
ch_m['cancel_rate'] = m.groupby(['channel', 'month']).agg(cancel_rate=('cancel', 'mean'))
ch_m

count  average_check  cancel_rate
channel  month                                   
органика 9        807   15133.110285     0.089165
         10       811   14160.320592     0.120390
реклама  9       1213   14868.079143     0.090705
         10      1194   14629.480737     0.116210
рефералы 9        352   14193.210227     0.073684
         10       362   15104.696133     0.110565

In [ ]:
# 3Какая доля заказов каждого канала из Шымкента? Объясняет ли канал падение?
shym = m[m['city']=='Шымкент']
shym.groupby('channel')['amount'].agg('count')

channel
органика    267
реклама     396
рефералы    113
Name: amount, dtype: int64

In [ ]:

other.groupby('channel')['amount'].agg('count')

KeyError: 'channel'